In [1]:
import pandas as pd
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, f1_score, confusion_matrix

df=pd.read_csv(r'C:\Users\ADMIN\OneDrive\Desktop\5thsem_OJT\project1_diabetes\data\processed\dilna_P1W1_features.csv')

feature_cols = ["age","pregnancies","glucose","blood_pressure","skin_thickness","insulin","bmi","diabetes_pedigree","bmi category","glucose category"]
x= pd.get_dummies(df[feature_cols],drop_first=True)
y=df["outcome"]

x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)




In [2]:
candidates = {
    "LogReg": ("scaled", LogisticRegression(max_iter=1000)),
    "Decision Tree": ("raw", DecisionTreeClassifier(max_depth=3, random_state=42)),
    "KNN": ("scaled", KNeighborsClassifier(n_neighbors=25)),
    "SVC balanced": ("scaled", SVC(class_weight="balanced", random_state=42)),
    "RF balanced": ("raw", RandomForestClassifier(
        max_depth=5, class_weight="balanced", random_state=42
    )),
}

TARGET_RECALL, GUARD_F1 = 0.70, 0.50  # Week 5 decision


print("rows:", len(df), "| train:", len(x_train), "| test:", len(x_test))
print("diabetic in test set:", int(y_test.sum()))
print("features after get_dummies:", x.shape[1])
print("candidates:", list(candidates))
print("target: recall >=", TARGET_RECALL, "with F1 >=", GUARD_F1)

rows: 950 | train: 760 | test: 190
diabetic in test set: 55
features after get_dummies: 13
candidates: ['LogReg', 'Decision Tree', 'KNN', 'SVC balanced', 'RF balanced']
target: recall >= 0.7 with F1 >= 0.5


Re-spIit the data and watch 

In [3]:
rows = []

for rs in [0, 1, 7, 42, 99]:
    Xa, Xb, ya, yb = train_test_split(
        x, y, test_size=0.2,
        random_state=rs, stratify=y
    )

    row = {"split": rs}

    for name, (kind, model) in candidates.items():
        m = clone(model)                       #   fresh copy

        if kind == "scaled":
            sc = StandardScaler().fit(Xa)
            m.fit(sc.transform(Xa), ya)
            pred = m.predict(sc.transform(Xb))
        else:
            m.fit(Xa, ya)
            pred = m.predict(Xb)
        row[name] = round(recall_score(yb, pred),3)
    rows.append(row)
    
splits = pd.DataFrame(rows).set_index("split")
print(splits)
spread = (splits.max()  - splits.min()).round(3)
print("\n",spread.sort_values(ascending=False))

       LogReg  Decision Tree    KNN  SVC balanced  RF balanced
split                                                         
0       0.218          0.091  0.127         0.600        0.600
1       0.255          0.382  0.164         0.745        0.618
7       0.255          0.182  0.200         0.655        0.582
42      0.291          0.291  0.182         0.618        0.655
99      0.273          0.236  0.200         0.709        0.655

 Decision Tree    0.291
SVC balanced     0.145
LogReg           0.073
KNN              0.073
RF balanced      0.073
dtype: float64


## K-Fold Cross-Validation

**K-Fold Cross-Validation** is a technique used to evaluate how well a Machine Learning model performs on different parts of the dataset.

### How it works

1. The dataset is divided into **K equal-sized folds**.
2. The model trains on K−1 folds and tests on the remaining fold.
3. This process repeats K times, with a different fold used for testing each time.
4. The final performance is calculated by averaging the results from all K runs.

In [4]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for i, (tr,te) in enumerate(cv.split(x_train,y_train),1):
    print(f"fold {i}: train {len(tr)} score{len(te)}"
          f"diabetic in score fol {y_train.iloc[te].sum()}")

def make_model(kind, model):
    """scaled models get the scaler INSIDE a Pipeline, so it refits per fold."""
    if kind == "scaled":
        return Pipeline([("scale", StandardScaler()),("model",model)])
    return model
print()

for name, (kind, model) in candidates.items():
    print(f"{name:15s} -> {type(make_model(kind, model)).__name__}")

fold 1: train 608 score152diabetic in score fol 43
fold 2: train 608 score152diabetic in score fol 43
fold 3: train 608 score152diabetic in score fol 44
fold 4: train 608 score152diabetic in score fol 44
fold 5: train 608 score152diabetic in score fol 44

LogReg          -> Pipeline
Decision Tree   -> DecisionTreeClassifier
KNN             -> Pipeline
SVC balanced    -> Pipeline
RF balanced     -> RandomForestClassifier


## cross validation
set-up cross validation in the right way

cross validate every candidate

In [6]:
from sklearn.model_selection import cross_validate

rows = []
for name, (kind, model ) in candidates.items():
    est = make_model(kind, model)
    res = cross_validate(est, x_train, y_train, cv=cv,
                         scoring=["recall","f1"])
    single = clone(est).fit(x_train, y_train)
    rows.append({
        "model": name,
        "cv_recall": round(res["test_recall"].mean(), 3),
        "cv_std": round(res["test_recall"].std(), 3),
        "cv_f1": round(res["test_f1"].mean(), 3),
        "weak6_recall": round(recall_score(y_test, single.predict(x_test)), 3 ),
        "folds": np.round(res["test_recall"],3).tolist(),
    })

cv_table = pd.DataFrame(rows).sort_values("cv_recall", ascending=False)
print(cv_table.to_string(index=False))

        model  cv_recall  cv_std  cv_f1  weak6_recall                               folds
 SVC balanced      0.721   0.046  0.559         0.618 [0.744, 0.791, 0.682, 0.727, 0.659]
  RF balanced      0.656   0.054  0.523         0.655 [0.721, 0.651, 0.705, 0.636, 0.568]
       LogReg      0.303   0.073  0.390         0.291  [0.419, 0.326, 0.318, 0.25, 0.205]
Decision Tree      0.216   0.076  0.296         0.291 [0.116, 0.326, 0.159, 0.205, 0.273]
          KNN      0.174   0.018  0.261         0.182 [0.186, 0.163, 0.159, 0.205, 0.159]


In [7]:
cv_table["low"] = (cv_table["cv_recall"] - cv_table["cv_std"]).round(3)
cv_table["high"] = (cv_table["cv_recall"] + cv_table["cv_std"]).round(3)
cv_table["clear_target"] = cv_table["cv_recall"] >= TARGET_RECALL
cv_table["clear_guard"] = cv_table["cv_f1"] >= GUARD_F1

print(cv_table[["model","cv_recall","low","high","cv_f1",
                "clear_target","clear_guard"]].to_string(index=False))

        model  cv_recall   low  high  cv_f1  clear_target  clear_guard
 SVC balanced      0.721 0.675 0.767  0.559          True         True
  RF balanced      0.656 0.602 0.710  0.523         False         True
       LogReg      0.303 0.230 0.376  0.390         False        False
Decision Tree      0.216 0.140 0.292  0.296         False        False
          KNN      0.174 0.156 0.192  0.261         False        False


## Hyperparameter Tuning

Hyperparameter tuning is the process of finding the best settings for a machine learning model to improve its performance.

### GridSearchCV
GridSearchCV tests every possible combination of the specified hyperparameter values using cross-validation and selects the best combination based on a scoring metric.

### RandomizedSearchCV
RandomizedSearchCV tests a specified number of randomly selected hyperparameter combinations. It is usually faster when the search space is large.

### Difference
- **GridSearchCV:** Tests all specified combinations.
- **RandomizedSearchCV:** Tests a limited number of randomly selected combinations.
- Both use cross-validation to evaluate model performance and help find suitable hyperparameters.

In [8]:
rf = RandomForestClassifier(random_state=42)
for setting, value in rf.get_params().items():
    print(f"{setting:25s} {value}")

bootstrap                 True
ccp_alpha                 0.0
class_weight              None
criterion                 gini
max_depth                 None
max_features              sqrt
max_leaf_nodes            None
max_samples               None
min_impurity_decrease     0.0
min_samples_leaf          1
min_samples_split         2
min_weight_fraction_leaf  0.0
monotonic_cst             None
n_estimators              100
n_jobs                    None
oob_score                 False
random_state              42
verbose                   0
warm_start                False


build the grid and count the post

In [9]:
param_grid ={
    "max_depth"          : [ 3,5,8,None],
    "min_samples_leaf": [1,5,10],
    "class_weight" : ["balanced","balanced_subsample"]
    }

combos = 1

for values in param_grid.values():
    combos *= len(values)

folds = cv.get_n_splits()
total_fits = combos*folds

print("combinations:", combos)
print("cross-validation folds:",folds)
print("Total model fits:",total_fits)

combinations: 24
cross-validation folds: 5
Total model fits: 120


In [10]:
from sklearn.model_selection import GridSearchCV

gs_recall = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=cv,
    scoring="recall",
    n_jobs=-1
)

gs_recall.fit(x_train, y_train)

print("Best settings:", gs_recall.best_params_)
print("Best CV recall:", round(gs_recall.best_score_,3))

top= pd.DataFrame(gs_recall.cv_results_)

print(
    top[["params","mean_test_score","std_test_score","rank_test_score"]]
    .sort_values("rank_test_score")
    .head(5)
    .to_string(index=False)
)

Best settings: {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1}
Best CV recall: 0.739
                                                                        params  mean_test_score  std_test_score  rank_test_score
           {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1}         0.738689        0.041181                1
           {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 5}         0.738584        0.041746                2
          {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 10}         0.734249        0.062548                3
{'class_weight': 'balanced_subsample', 'max_depth': 3, 'min_samples_leaf': 10}         0.720719        0.064830                4
 {'class_weight': 'balanced_subsample', 'max_depth': 3, 'min_samples_leaf': 5}         0.711416        0.064128                5


Test the tuned model on unscreen patients

In [11]:
best_rf = gs_recall.best_estimator_
pred = best_rf.predict(x_test)
tn, fp, fn, tp = confusion_matrix(y_test,pred).ravel()

print("Search CV recall:",round(gs_recall.best_score_,3))
print("Test recall     :",round(recall_score(y_test, pred), 3))
print("Test F1         :",round(f1_score(y_test, pred), 3))

print(f"Diabetic patients found: {tp} of {tp + fn}")
print(f"Missed patients        :{fn}")
print(f"False alarms         :{fp}")

Search CV recall: 0.739
Test recall     : 0.745
Test F1         : 0.536
Diabetic patients found: 41 of 55
Missed patients        :14
False alarms         :57


Change the scoring, change the winner

In [12]:
from sklearn.model_selection import GridSearchCV

gs_f1= GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

gs_f1.fit(x_train, y_train)

for label, gs in[
    ("Searched on recall", gs_recall),
    ("Searched on F1", gs_f1)
]:
    pred = gs.best_estimator_.predict(x_test)
    found = int(((pred == 1) & (y_test.values == 1)).sum())
    
    print(label,":", gs.best_params_)
    print("  Test recall:", round(recall_score(y_test,pred), 3))
    print(" Test F1     :",round(f1_score(y_test,pred), 3))
    print("patients found:", found)
    print()



Searched on recall : {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1}
  Test recall: 0.745
 Test F1     : 0.536
patients found: 41

Searched on F1 : {'class_weight': 'balanced', 'max_depth': 5, 'min_samples_leaf': 10}
  Test recall: 0.655
 Test F1     : 0.522
patients found: 36



Tune the runner-up with s random search

In [13]:
from sklearn.model_selection import RandomizedSearchCV

svc_Pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", SVC(class_weight="balanced",random_state=42))
    ])
svc_grid = {
    "model_c":[0.1, 0.5, 1, 5, 10, 50],
    "model_gamma" : ["scale", 0.005, 0.01, 0.05, 0.1] 
}

rs_svc = RandomizedSearchCV(
    svc_Pipe, svc_grid,
    n_iter=15,  cv=cv,
    scoring = "recall",
    random_state=42, n_jobs=-1
)
rs_svc.fit(x_train, y_train)



svc_pred = rs_svc.best_estimator_.predict(x_test)
svc_test = recall_score(y_test, svc_pred)

ValueError: Invalid parameter 'model_gamma' for estimator Pipeline(steps=[('scale', StandardScaler()),
                ('model', SVC(class_weight='balanced', random_state=42))]). Valid parameters are: ['memory', 'steps', 'transform_input', 'verbose'].